# LeetCode 2166 — Design Bitset bằng Python

Link đề: https://leetcode.com/problems/design-bitset/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String, Design

Skeleton LeetCode:

```python
class Bitset:

    def __init__(self, size: int):
        ...

    def fix(self, idx: int) -> None:
        ...

    def unfix(self, idx: int) -> None:
        ...

    def flip(self) -> None:
        ...

    def all(self) -> bool:
        ...

    def one(self) -> bool:
        ...

    def count(self) -> int:
        ...

    def toString(self) -> str:
        ...
```

Bài **thiết kế cấu trúc dữ liệu**: mọi thao tác phải nhanh, đặc biệt `flip` (đảo **toàn bộ** bit) và `count`. Notebook xây **bộ chạy lệnh** kiểu LeetCode, đối chiếu mọi cài đặt với bản **ngây thơ** trên chuỗi lệnh ngẫu nhiên, đi từ **mảng bit, `flip` `O(n)`** → **cờ đảo lười (lazy flip) + bộ đếm** → **hai mảng bù nhau, `flip` = đổi tham chiếu** → **số nguyên lớn của Python làm bitset** → **bitset theo khối 64 bit**.

## 1. Mục tiêu học

1. Thiết kế lớp với **bất biến** rõ ràng: `cnt` luôn bằng số bit 1 **thực sự**, kể cả sau `flip`.
2. Kỹ thuật **lười (lazy)**: thay vì đảo `n` bit, lưu một cờ `flipped` và **diễn giải lại** bit vật lý khi đọc/ghi.
3. Kỹ thuật **lưu cả hai trạng thái** (bitset và phần bù) để `flip` chỉ là đổi tham chiếu `O(1)`.
4. Dùng số nguyên Python như một bitset: `|`, `& ~`, `^ mask`, `int.bit_count()`.
5. Viết **command runner** và kiểm thử thiết kế bằng đối chiếu ngẫu nhiên với bản ngây thơ.

## 2. Đề bài & ví dụ

Cài đặt lớp `Bitset`:

- `Bitset(size)`: khởi tạo `size` bit, tất cả bằng 0.
- `fix(idx)`: đặt bit `idx` thành 1 (đã là 1 thì không đổi).
- `unfix(idx)`: đặt bit `idx` thành 0 (đã là 0 thì không đổi).
- `flip()`: đảo **mọi** bit (0 ↔ 1).
- `all()`: `True` nếu **mọi** bit là 1.
- `one()`: `True` nếu có **ít nhất một** bit 1.
- `count()`: số bit 1.
- `toString()`: chuỗi `size` ký tự, ký tự thứ `i` là giá trị bit `i`.

**Ví dụ 1:**
```
["Bitset","fix","fix","flip","all","unfix","flip","one","unfix","count","toString"]
[[5],[3],[1],[],[],[0],[],[],[0],[],[]]
→ [null,null,null,null,false,null,null,true,null,2,"01010"]
```

**Ràng buộc:** `1 ≤ size ≤ 10⁵`, `0 ≤ idx < size`, tối đa `10⁵` lời gọi tổng cộng, **tối đa 5** lời gọi `toString`.

In [1]:
import random
import time
from typing import List

EX_NAMES = ["Bitset", "fix", "fix", "flip", "all", "unfix", "flip", "one", "unfix", "count", "toString"]
EX_ARGS = [[5], [3], [1], [], [], [0], [], [], [0], [], []]
EX_OUT = [None, None, None, None, False, None, None, True, None, 2, "01010"]

def run_leetcode(cls, names, args):
    obj = cls(*args[0]); out = [None]
    for name, a in zip(names[1:], args[1:]):
        out.append(getattr(obj, name)(*a))
    return out

print(list(zip(EX_NAMES, EX_ARGS, EX_OUT)))

[('Bitset', [5], None), ('fix', [3], None), ('fix', [1], None), ('flip', [], None), ('all', [], False), ('unfix', [0], None), ('flip', [], None), ('one', [], True), ('unfix', [0], None), ('count', [], 2), ('toString', [], '01010')]


## 3. Kiến thức nền

### 3.1 Mảng bit bằng `bytearray`

`bytearray(n)` là mảng `n` byte bằng 0, sửa được tại chỗ, nhỏ gọn hơn `list` số nguyên.

In [2]:
b = bytearray(5)
b[3] = 1; b[1] = 1
print(list(b), "| chuỗi:", "".join(map(str, b)), "| hoặc:", b.translate(bytes.maketrans(b"\x00\x01", b"01")).decode())

[0, 1, 0, 1, 0] | chuỗi: 01010 | hoặc: 01010


### 3.2 Phép toán bit trên số nguyên Python

- đặt bit `i`: `x |= 1 << i`; xóa bit `i`: `x &= ~(1 << i)`; đọc bit: `x >> i & 1`;
- đảo `n` bit thấp: `x ^= (1 << n) − 1`;
- đếm bit 1: `x.bit_count()` (Python ≥ 3.10) hoặc `bin(x).count("1")`.

In [3]:
n = 5; x = 0
x |= 1 << 3; x |= 1 << 1
print("x =", bin(x), "| bit 3 =", x >> 3 & 1, "| đếm =", x.bit_count())
x ^= (1 << n) - 1
print("sau flip:", format(x, f"0{n}b"), "(chú ý: in theo thứ tự bit cao -> thấp, ngược với toString)")

x = 0b1010 | bit 3 = 1 | đếm = 2
sau flip: 10101 (chú ý: in theo thứ tự bit cao -> thấp, ngược với toString)


### 3.3 Bất biến (invariant) trong bài thiết kế

Giữ thêm **trạng thái phụ** (`cnt`, `flipped`, …) giúp truy vấn `O(1)`, nhưng **mọi** thao tác thay đổi dữ liệu phải cập nhật trạng thái phụ cho đúng. Cách kiểm tra tốt nhất: viết hàm `check_invariant(obj)` và gọi sau mỗi lệnh trong test.

### 3.4 Đổi tham chiếu hai biến trong `O(1)`

In [4]:
a, b = [0, 1, 0], [1, 0, 1]
a, b = b, a                      # chỉ đổi tham chiếu, không sao chép
print(a, b)

[1, 0, 1] [0, 1, 0]


## 4. Ý tưởng nền tảng / trực giác

Mô phỏng ví dụ bằng tay với chuỗi bit:

| lệnh | bitset | ghi chú |
|---|---|---|
| `Bitset(5)` | `00000` | |
| `fix(3)` | `00010` | |
| `fix(1)` | `01010` | |
| `flip()` | `10101` | đảo tất cả |
| `all()` | `10101` | `false` |
| `unfix(0)` | `00101` | |
| `flip()` | `11010` | |
| `one()` | `11010` | `true` |
| `unfix(0)` | `01010` | |
| `count()` | `01010` | `2` |
| `toString()` | `01010` | `"01010"` |

Câu hỏi thiết kế: `flip` đảo `n` bit — nếu làm thật sẽ tốn `O(n)`, với `10⁵` lời gọi và `n = 10⁵` là `10¹⁰` — quá chậm. Quan sát: sau `k` lần `flip`, bit thật = bit vật lý `XOR (k mod 2)`. Chỉ cần nhớ **tính chẵn lẻ** của số lần `flip`!

Và `count` sau `flip` = `size − count` cũ. ⇒ `all` ⇔ `cnt == size`, `one` ⇔ `cnt > 0`.

## 5. Cách 1 — Brute Force / baseline: mảng bit, làm đúng nghĩa đen

In [5]:
class BitsetNaive:
    def __init__(self, size: int):
        self.bits = [0] * size

    def fix(self, idx: int) -> None:
        self.bits[idx] = 1

    def unfix(self, idx: int) -> None:
        self.bits[idx] = 0

    def flip(self) -> None:
        self.bits = [1 - b for b in self.bits]       # O(n)

    def all(self) -> bool:
        return all(self.bits)                         # O(n)

    def one(self) -> bool:
        return any(self.bits)                         # O(n)

    def count(self) -> int:
        return sum(self.bits)                         # O(n)

    def toString(self) -> str:
        return "".join(map(str, self.bits))

out = run_leetcode(BitsetNaive, EX_NAMES, EX_ARGS)
print(out, "| khớp:", out == EX_OUT)

[None, None, None, None, False, None, None, True, None, 2, '01010'] | khớp: True


**Độ phức tạp:** `fix`/`unfix` `O(1)`; `flip`, `all`, `one`, `count`, `toString` đều `O(n)`. Tệ nhất `10⁵ × 10⁵ = 10¹⁰` — TLE. Rất **dễ thấy đúng** ⇒ dùng làm chuẩn đối chiếu.

**Nút thắt:** `flip` và `count`. Giải quyết bằng **cờ lười** + **bộ đếm**.

## 6. Ý tưởng thuật toán / bước đột phá

**Bất biến:** bit thật `i` = `phys[i] XOR flipped`; `cnt` = số bit thật bằng 1.

- `fix(i)`: bit thật muốn là 1 ⇒ bit vật lý cần là `1 XOR flipped`. **Nếu** bit thật đang là 0 **thì** ghi và `cnt += 1`; **ngược lại** không làm gì.
- `unfix(i)`: đối xứng, `cnt −= 1` khi bit thật đang là 1.
- `flip()`: `flipped ^= 1`, `cnt = size − cnt` — `O(1)`.
- `all()`: `cnt == size`; `one()`: `cnt > 0`; `count()`: `cnt`.
- `toString()`: diễn giải lại từng bit — `O(n)`, nhưng đề chỉ gọi tối đa 5 lần.

**Phương án khác:** lưu **cả** `bits` và `inv` (luôn là phần bù của nhau). `fix`/`unfix` sửa cả hai mảng `O(1)`; `flip` = đổi tham chiếu `bits, inv = inv, bits` — `O(1)`, và `toString` không cần diễn giải lại.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Cờ đảo lười + bộ đếm

In [6]:
class BitsetLazy:
    def __init__(self, size: int):
        self.n = size
        self.phys = bytearray(size)      # bit vật lý
        self.flipped = 0                 # bit thật = phys ^ flipped
        self.cnt = 0                     # số bit thật bằng 1

    def _get(self, idx):
        return self.phys[idx] ^ self.flipped

    def fix(self, idx: int) -> None:
        if not self._get(idx):
            self.phys[idx] ^= 1
            self.cnt += 1

    def unfix(self, idx: int) -> None:
        if self._get(idx):
            self.phys[idx] ^= 1
            self.cnt -= 1

    def flip(self) -> None:
        self.flipped ^= 1
        self.cnt = self.n - self.cnt

    def all(self) -> bool:
        return self.cnt == self.n

    def one(self) -> bool:
        return self.cnt > 0

    def count(self) -> int:
        return self.cnt

    def toString(self) -> str:
        f = self.flipped
        return "".join("1" if b ^ f else "0" for b in self.phys)

out = run_leetcode(BitsetLazy, EX_NAMES, EX_ARGS)
print(out, "| khớp:", out == EX_OUT)

[None, None, None, None, False, None, None, True, None, 2, '01010'] | khớp: True


**Độ phức tạp:** mọi thao tác `O(1)` trừ `toString` `O(n)`. Bộ nhớ `O(n)` byte.

### 7.2 Cách 3 — Hai mảng bù nhau, `flip` = đổi tham chiếu

In [7]:
class BitsetTwoArrays:
    def __init__(self, size: int):
        self.n = size
        self.bits = bytearray(b"0" * size)     # lưu thẳng ký tự '0'/'1' -> toString nhanh
        self.inv = bytearray(b"1" * size)      # luôn là phần bù của bits
        self.cnt = 0

    def fix(self, idx: int) -> None:
        if self.bits[idx] == 48:               # ord('0')
            self.bits[idx] = 49; self.inv[idx] = 48
            self.cnt += 1

    def unfix(self, idx: int) -> None:
        if self.bits[idx] == 49:
            self.bits[idx] = 48; self.inv[idx] = 49
            self.cnt -= 1

    def flip(self) -> None:
        self.bits, self.inv = self.inv, self.bits
        self.cnt = self.n - self.cnt

    def all(self) -> bool:
        return self.cnt == self.n

    def one(self) -> bool:
        return self.cnt > 0

    def count(self) -> int:
        return self.cnt

    def toString(self) -> str:
        return self.bits.decode()

out = run_leetcode(BitsetTwoArrays, EX_NAMES, EX_ARGS)
print(out, "| khớp:", out == EX_OUT)

[None, None, None, None, False, None, None, True, None, 2, '01010'] | khớp: True


**Độ phức tạp:** mọi thao tác `O(1)`, `toString` là một lần `decode` `O(n)` chạy trong C (rất nhanh). Bộ nhớ `2n` byte. Mẹo lưu sẵn ký tự `'0'`/`'1'` (mã 48/49) giúp `toString` không cần chuyển đổi.

### 7.3 Cách 4 — Số nguyên lớn của Python làm bitset

Bit `i` của số `x` là bit thứ `i` của bitset. `flip` = `x ^= mask` với `mask = (1 << n) − 1`. Phép XOR trên số `n` bit tốn `O(n / 30)` (Python lưu số lớn theo "chữ số" 30 bit) — với `n = 10⁵` khoảng 3 300 phép máy, nhanh trong thực tế nhưng **không** `O(1)`. Kết hợp với `cnt` để `count` `O(1)`.

In [8]:
class BitsetBigInt:
    def __init__(self, size: int):
        self.n = size
        self.mask = (1 << size) - 1
        self.x = 0
        self.cnt = 0

    def fix(self, idx: int) -> None:
        if not (self.x >> idx & 1):
            self.x |= 1 << idx
            self.cnt += 1

    def unfix(self, idx: int) -> None:
        if self.x >> idx & 1:
            self.x &= ~(1 << idx)
            self.cnt -= 1

    def flip(self) -> None:
        self.x ^= self.mask
        self.cnt = self.n - self.cnt

    def all(self) -> bool:
        return self.cnt == self.n

    def one(self) -> bool:
        return self.cnt > 0

    def count(self) -> int:
        return self.cnt

    def toString(self) -> str:
        return format(self.x, f"0{self.n}b")[::-1]     # bit 0 đứng đầu chuỗi

out = run_leetcode(BitsetBigInt, EX_NAMES, EX_ARGS)
print(out, "| khớp:", out == EX_OUT)

[None, None, None, None, False, None, None, True, None, 2, '01010'] | khớp: True


**Cảnh báo hiệu năng:** `self.x |= 1 << idx` tạo một số mới `O(n/30)` mỗi lần (số nguyên Python **bất biến**) — nên `fix`/`unfix` cũng không `O(1)`. Đây là cách "gọn" nhưng hằng số phụ thuộc `n`; xem số đo ở mục 11.

### 7.4 Cách 5 — Bitset theo khối 64 bit (như `std::bitset` / `java.util.BitSet`)

Chia `n` bit thành `⌈n/64⌉` từ (word) 64 bit trong một `list`; mỗi `fix`/`unfix` chỉ sửa **một** word. Kết hợp cờ lười cho `flip` và `cnt` cho `count`. Đây là cách các thư viện thật làm (để tiết kiệm bộ nhớ 8 lần so với `bytearray`).

In [9]:
class BitsetWords:
    W = 64
    def __init__(self, size: int):
        self.n = size
        self.words = [0] * ((size + 63) // 64)
        self.flipped = 0
        self.cnt = 0

    def _get(self, idx):
        return (self.words[idx >> 6] >> (idx & 63) & 1) ^ self.flipped

    def _toggle(self, idx):
        self.words[idx >> 6] ^= 1 << (idx & 63)

    def fix(self, idx: int) -> None:
        if not self._get(idx):
            self._toggle(idx); self.cnt += 1

    def unfix(self, idx: int) -> None:
        if self._get(idx):
            self._toggle(idx); self.cnt -= 1

    def flip(self) -> None:
        self.flipped ^= 1
        self.cnt = self.n - self.cnt

    def all(self) -> bool:
        return self.cnt == self.n

    def one(self) -> bool:
        return self.cnt > 0

    def count(self) -> int:
        return self.cnt

    def toString(self) -> str:
        out = []
        for w in self.words:
            out.append(format(w, "064b")[::-1])
        s = "".join(out)[: self.n]
        if self.flipped:
            s = s.translate(str.maketrans("01", "10"))
        return s

out = run_leetcode(BitsetWords, EX_NAMES, EX_ARGS)
print(out, "| khớp:", out == EX_OUT)

[None, None, None, None, False, None, None, True, None, 2, '01010'] | khớp: True


**Độ phức tạp:** mọi thao tác `O(1)` (trừ `toString` `O(n)`), bộ nhớ `n/64` word.

## 8. TRACE từng bước

### 8.1 Trace cách 2 (cờ lười) trên ví dụ của đề — in bit vật lý, cờ, bit thật, `cnt`

In [10]:
def trace_lazy(names, args):
    obj = BitsetLazy(*args[0])
    print(f"  {'lệnh':<12} {'kết quả':<8} {'phys':<7} {'flipped':<8} {'thật':<7} cnt")
    for name, a in zip(names[1:], args[1:]):
        r = getattr(obj, name)(*a)
        phys = "".join(map(str, obj.phys))
        real = "".join(str(b ^ obj.flipped) for b in obj.phys)
        call = f"{name}({','.join(map(str, a))})"
        print(f"  {call:<12} {str(r):<8} {phys:<7} {obj.flipped:<8} {real:<7} {obj.cnt}")

trace_lazy(EX_NAMES, EX_ARGS)

  lệnh         kết quả  phys    flipped  thật    cnt
  fix(3)       None     00010   0        00010   1
  fix(1)       None     01010   0        01010   2
  flip()       None     01010   1        10101   3
  all()        False    01010   1        10101   3
  unfix(0)     None     11010   1        00101   2
  flip()       None     11010   0        11010   3
  one()        True     11010   0        11010   3
  unfix(0)     None     01010   0        01010   2
  count()      2        01010   0        01010   2
  toString()   01010    01010   0        01010   2


Chú ý dòng `unfix(0)` ngay sau `flip()` đầu tiên: bit thật 0 đang là `1` (vật lý `0` XOR cờ `1`), nên ta **đổi bit vật lý thành 1** — nghe ngược đời nhưng đúng: bit thật = `1 XOR 1 = 0`.

### 8.2 Trace cách 3 (hai mảng) trên chuỗi lệnh tự dựng (đã đối chiếu bản ngây thơ — đối chiếu lại đề gốc)

In [11]:
def trace_two_arrays(size, ops):
    obj = BitsetTwoArrays(size); ref = BitsetNaive(size)
    for name, a in ops:
        r = getattr(obj, name)(*a); r2 = getattr(ref, name)(*a)
        assert r == r2
        print(f"  {name + str(tuple(a)):<12} -> {str(r):<6} bits={obj.bits.decode()} inv={obj.inv.decode()} cnt={obj.cnt}")

trace_two_arrays(4, [("flip", []), ("unfix", [2]), ("all", []), ("fix", [2]), ("all", []), ("flip", []), ("one", []), ("fix", [0]), ("count", []), ("toString", [])])

  flip()       -> None   bits=1111 inv=0000 cnt=4
  unfix(2,)    -> None   bits=1101 inv=0010 cnt=3
  all()        -> False  bits=1101 inv=0010 cnt=3
  fix(2,)      -> None   bits=1111 inv=0000 cnt=4
  all()        -> True   bits=1111 inv=0000 cnt=4
  flip()       -> None   bits=0000 inv=1111 cnt=0
  one()        -> False  bits=0000 inv=1111 cnt=0
  fix(0,)      -> None   bits=1000 inv=0111 cnt=1
  count()      -> 1      bits=1000 inv=0111 cnt=1
  toString()   -> 1000   bits=1000 inv=0111 cnt=1


## 9. Code nộp LeetCode

In [12]:
class Bitset:

    def __init__(self, size: int):
        self.n = size
        self.bits = bytearray(b"0" * size)   # trạng thái hiện tại (ký tự '0'/'1')
        self.inv = bytearray(b"1" * size)    # phần bù, dùng cho flip O(1)
        self.cnt = 0                         # số bit 1 hiện tại

    def fix(self, idx: int) -> None:
        if self.bits[idx] == 48:
            self.bits[idx] = 49
            self.inv[idx] = 48
            self.cnt += 1

    def unfix(self, idx: int) -> None:
        if self.bits[idx] == 49:
            self.bits[idx] = 48
            self.inv[idx] = 49
            self.cnt -= 1

    def flip(self) -> None:
        self.bits, self.inv = self.inv, self.bits
        self.cnt = self.n - self.cnt

    def all(self) -> bool:
        return self.cnt == self.n

    def one(self) -> bool:
        return self.cnt > 0

    def count(self) -> int:
        return self.cnt

    def toString(self) -> str:
        return self.bits.decode()

out = run_leetcode(Bitset, EX_NAMES, EX_ARGS)
print(out)
assert out == EX_OUT
print("Ví dụ đề: OK")

[None, None, None, None, False, None, None, True, None, 2, '01010']
Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Cách 3 / code nộp.** Bất biến: (I1) `bits` là trạng thái thật; (I2) `inv[i] = 1 − bits[i]` với mọi `i`; (I3) `cnt` = số `'1'` trong `bits`.
- Khởi tạo: `bits` toàn `'0'`, `inv` toàn `'1'`, `cnt = 0` ✓.
- `fix(i)`: chỉ đổi khi `bits[i] = '0'`: đặt `bits[i] = '1'`, `inv[i] = '0'` (giữ I2), `cnt += 1` (giữ I3). Khi đã là `'1'`: không đổi gì ✓. `unfix` đối xứng.
- `flip()`: trạng thái thật mới = phần bù của trạng thái cũ = `inv` (theo I2) ⇒ đổi tham chiếu giữ I1; mảng `inv` mới là `bits` cũ, đúng là phần bù ⇒ I2; số bit 1 mới = `n − cnt` ⇒ I3.
- Truy vấn đọc trực tiếp từ bất biến.

**Cách 2.** Bất biến: bit thật `= phys ^ flipped`, `cnt` = số bit thật bằng 1. `fix`/`unfix` đổi `phys[i]` đúng một lần khi bit thật cần đổi; `flip` đổi `flipped` (mọi bit thật đảo) và `cnt = n − cnt`.

**Cách 4, 5** dùng cùng lập luận với số nguyên/words thay cho mảng.

## 11. Phân tích Time / Space Complexity

Gọi `n = size`, `q` = số lời gọi.

| Cách | `fix`/`unfix` | `flip` | `all`/`one`/`count` | `toString` | Space |
|---|---|---|---|---|---|
| 1. Ngây thơ | `O(1)` | `O(n)` | `O(n)` | `O(n)` | `O(n)` |
| 2. Cờ lười | `O(1)` | `O(1)` | `O(1)` | `O(n)` | `n` byte |
| 3. Hai mảng | `O(1)` | `O(1)` | `O(1)` | `O(n)` (C) | `2n` byte |
| 4. Số nguyên lớn | `O(n/30)` | `O(n/30)` | `O(1)` | `O(n)` | `n/8` byte |
| 5. Khối 64 bit | `O(1)` | `O(1)` | `O(1)` | `O(n)` | `n/8` byte (+ overhead list) |

In [13]:
def random_ops(size, q, rnd, max_tostring=5):
    ops = []; ts = 0
    for _ in range(q):
        r = rnd.random()
        if r < 0.3: ops.append(("fix", [rnd.randrange(size)]))
        elif r < 0.6: ops.append(("unfix", [rnd.randrange(size)]))
        elif r < 0.75: ops.append(("flip", []))
        elif r < 0.82: ops.append(("all", []))
        elif r < 0.89: ops.append(("one", []))
        elif r < 0.99 or ts >= max_tostring: ops.append(("count", []))
        else: ops.append(("toString", [])); ts += 1
    return ops

def run(cls, size, ops):
    obj = cls(size)
    return [getattr(obj, name)(*a) for name, a in ops]

rnd = random.Random(2166)
size, q = 10**5, 10**5
ops = random_ops(size, q, rnd)
for cls in [BitsetLazy, BitsetTwoArrays, BitsetBigInt, BitsetWords, Bitset]:
    t0 = time.perf_counter(); run(cls, size, ops); dt = time.perf_counter() - t0
    print(f"{cls.__name__:<16}: {q} lệnh, size={size}: {dt * 1000:8.1f} ms")
small_ops = random_ops(size, 300, rnd)
t0 = time.perf_counter(); run(BitsetNaive, size, small_ops)
print(f"{'BitsetNaive':<16}: chỉ 300 lệnh: {(time.perf_counter() - t0) * 1000:8.1f} ms  (10^5 lệnh sẽ chậm hơn ~300 lần)")

BitsetLazy      : 100000 lệnh, size=100000:     26.6 ms
BitsetTwoArrays : 100000 lệnh, size=100000:     11.7 ms


BitsetBigInt    : 100000 lệnh, size=100000:    122.2 ms
BitsetWords     : 100000 lệnh, size=100000:     22.9 ms
Bitset          : 100000 lệnh, size=100000:     11.1 ms


BitsetNaive     : chỉ 300 lệnh:     80.9 ms  (10^5 lệnh sẽ chậm hơn ~300 lần)


## 12. Bộ test

In [14]:
IMPLS = [BitsetLazy, BitsetTwoArrays, BitsetBigInt, BitsetWords, Bitset]

def check(size, ops, exp=None):
    ref = run(BitsetNaive, size, ops)
    for cls in IMPLS:
        got = run(cls, size, ops)
        assert got == ref, (cls.__name__, size, ops, got, ref)
    if exp is not None:
        assert ref == exp
    return ref

for cls in [BitsetNaive] + IMPLS:
    assert run_leetcode(cls, EX_NAMES, EX_ARGS) == EX_OUT
print("Ví dụ đề (mọi cài đặt): OK")

check(1, [("all", []), ("one", []), ("flip", []), ("all", []), ("one", []), ("toString", [])], [False, False, None, True, True, "1"])
check(3, [("fix", [1]), ("fix", [1]), ("count", [])], [None, None, 1])                  # fix hai lần
check(3, [("unfix", [1]), ("count", [])], [None, 0])                                   # unfix bit đang 0
check(3, [("flip", []), ("flip", []), ("toString", [])], [None, None, "000"])           # flip hai lần
check(4, [("flip", []), ("unfix", [2]), ("fix", [2]), ("all", [])], [None, None, None, True])
check(70, [("fix", [65]), ("flip", []), ("count", []), ("unfix", [3]), ("toString", [])])   # vượt biên một word 64 bit
print("Edge case: OK")

rnd = random.Random(2021)
for _ in range(2000):
    size = rnd.randint(1, 10)
    check(size, random_ops(size, rnd.randint(1, 30), rnd, max_tostring=30))
print("2000 chuỗi lệnh ngẫu nhiên (size nhỏ, nhiều toString): OK")

for _ in range(30):
    size = rnd.randint(60, 200)
    check(size, random_ops(size, 500, rnd, max_tostring=5))
print("30 chuỗi 500 lệnh (size quanh biên word): OK")

Ví dụ đề (mọi cài đặt): OK
Edge case: OK
2000 chuỗi lệnh ngẫu nhiên (size nhỏ, nhiều toString): OK
30 chuỗi 500 lệnh (size quanh biên word): OK


In [15]:
# Kiểm bất biến của code nộp sau MỖI lệnh
def check_invariant(b):
    s = b.bits.decode(); t = b.inv.decode()
    assert all(x != y for x, y in zip(s, t)), "inv phải là phần bù của bits"
    assert b.cnt == s.count("1"), "cnt phải bằng số bit 1"

rnd = random.Random(77)
for _ in range(300):
    size = rnd.randint(1, 12); b = Bitset(size)
    for name, a in random_ops(size, 40, rnd, max_tostring=40):
        getattr(b, name)(*a); check_invariant(b)
print("Bất biến (bits/inv bù nhau, cnt đúng) luôn giữ trên 300 chuỗi lệnh")

Bất biến (bits/inv bù nhau, cnt đúng) luôn giữ trên 300 chuỗi lệnh


## 13. Các lỗi người mới hay gặp

Mỗi lỗi là một lớp con của cài đặt đúng, chỉ sửa **một** chỗ, chạy trên chuỗi lệnh ngắn.

### Lỗi 1 — Cờ lười nhưng `fix` ghi thẳng bit vật lý = 1 (quên cờ)

In [16]:
class BugLazyFix(BitsetLazy):
    def fix(self, idx):
        if not self._get(idx):
            self.phys[idx] = 1                       # SAI: phải là 1 ^ flipped
            self.cnt += 1

ops = [("flip", []), ("unfix", [0]), ("fix", [0]), ("toString", [])]
print("sai :", run(BugLazyFix, 2, ops), "| đúng:", run(Bitset, 2, ops))

sai : [None, None, None, '01'] | đúng: [None, None, None, '11']


**Vì sao sai:** khi `flipped = 1`, bit vật lý 1 nghĩa là bit thật **0**. `cnt` vẫn tăng nên `count` có vẻ đúng nhưng `toString` sai — và hai trạng thái lệch nhau. **Sửa:** `phys[idx] ^= 1` (đảo bit vật lý), hoặc ghi `1 ^ flipped`.

### Lỗi 2 — `flip` quên cập nhật `cnt`

In [17]:
class BugFlipCount(BitsetLazy):
    def flip(self):
        self.flipped ^= 1                            # SAI: thiếu cnt = n - cnt

ops = [("fix", [0]), ("flip", []), ("count", []), ("all", [])]
print("sai :", run(BugFlipCount, 3, ops), "| đúng:", run(Bitset, 3, ops))

sai : [None, None, 1, False] | đúng: [None, None, 2, False]


**Sửa:** `self.cnt = self.n − self.cnt`.

### Lỗi 3 — `fix` tăng `cnt` cả khi bit đã là 1

In [18]:
class BugDoubleFix(BitsetTwoArrays):
    def fix(self, idx):
        self.bits[idx] = 49; self.inv[idx] = 48
        self.cnt += 1                                # SAI: không kiểm tra bit cũ

ops = [("fix", [1]), ("fix", [1]), ("count", []), ("all", [])]
print("sai :", run(BugDoubleFix, 2, ops), "| đúng:", run(Bitset, 2, ops))

sai : [None, None, 2, True] | đúng: [None, None, 1, False]


**Vì sao sai:** `cnt = 2` trên bitset `"01"` ⇒ `all()` tưởng mọi bit là 1. **Sửa:** chỉ tăng khi bit đang là 0 — đề nói rõ "nếu đã là 1 thì không đổi".

### Lỗi 4 — Cờ lười nhưng `toString` in bit vật lý

In [19]:
class BugToString(BitsetLazy):
    def toString(self):
        return "".join(map(str, self.phys))          # SAI: quên XOR với flipped

ops = [("fix", [0]), ("flip", []), ("toString", [])]
print("sai :", run(BugToString, 3, ops), "| đúng:", run(Bitset, 3, ops))

sai : [None, None, '100'] | đúng: [None, None, '011']


**Sửa:** `"1" if b ^ flipped else "0"`.

### Lỗi 5 — Hai mảng nhưng `fix` chỉ sửa `bits`, quên `inv`

In [20]:
class BugForgetInv(BitsetTwoArrays):
    def fix(self, idx):
        if self.bits[idx] == 48:
            self.bits[idx] = 49                      # SAI: inv[idx] vẫn là '1'
            self.cnt += 1

ops = [("fix", [0]), ("flip", []), ("toString", []), ("count", [])]
print("sai :", run(BugForgetInv, 2, ops), "| đúng:", run(Bitset, 2, ops))

sai : [None, None, '11', 1] | đúng: [None, None, '01', 1]


**Vì sao sai:** bất biến "`inv` là phần bù" bị phá; lỗi chỉ lộ ra **sau** `flip` (`toString` cho `"11"` trong khi `count` nói 1). **Bài học:** viết `check_invariant` như mục 12 — nó bắt lỗi ngay lệnh đầu tiên.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thao tác "áp dụng cho **toàn bộ**" (đảo, cộng thêm, gán) ⇒ nghĩ ngay tới **cờ lười / giá trị offset toàn cục**.
- Truy vấn tổng hợp (`count`, `all`, `one`) ⇒ duy trì **bộ đếm** cập nhật ở mọi thao tác ghi.
- Muốn `flip` `O(1)` **và** `toString` nhanh ⇒ giữ cả trạng thái lẫn phần bù.
- Đọc kỹ giới hạn đặc biệt ("tối đa 5 lần `toString`") — cho phép thao tác `O(n)` ở đó.
- Kiểm thử thiết kế: **bản ngây thơ + runner + chuỗi lệnh ngẫu nhiên + hàm bất biến**.

## 15. Biến thể của bài

1. **`flip(l, r)` đảo một đoạn**, `count(l, r)` đếm trên đoạn ⇒ **segment tree với lazy propagation** (bài tập 5).
2. **LeetCode 1622 — Fancy Sequence:** "cộng/nhân toàn bộ" lười bằng biến đổi affine toàn cục.
3. **LeetCode 2569 — Handling Sum Queries After Update:** đảo đoạn bit + truy vấn tổng (segment tree lazy).
4. **`set_all(v)`** (gán toàn bộ bằng `v`) ⇒ lưu "phiên bản" (timestamp) của lần gán toàn cục.
5. **Bitset động** cho bài toán knapsack (`dp |= dp << w`) — dùng số nguyên Python như cách 4.

In [21]:
class SegFlip:
    """Biến thể 1: đảo đoạn [l, r], đếm bit 1 trên [l, r] — segment tree + lazy (iterative-friendly, depth ~ log n)."""
    def __init__(self, n):
        self.n = n; self.cnt = [0] * (4 * n); self.lz = [0] * (4 * n)
    def _apply(self, node, length):
        self.cnt[node] = length - self.cnt[node]; self.lz[node] ^= 1
    def _push(self, node, l, m, r):
        if self.lz[node]:
            self._apply(2 * node, m - l + 1); self._apply(2 * node + 1, r - m); self.lz[node] = 0
    def flip(self, ql, qr, node=1, l=0, r=None):
        if r is None: r = self.n - 1
        if qr < l or r < ql: return
        if ql <= l and r <= qr:
            self._apply(node, r - l + 1); return
        m = (l + r) // 2; self._push(node, l, m, r)
        self.flip(ql, qr, 2 * node, l, m); self.flip(ql, qr, 2 * node + 1, m + 1, r)
        self.cnt[node] = self.cnt[2 * node] + self.cnt[2 * node + 1]
    def count(self, ql, qr, node=1, l=0, r=None):
        if r is None: r = self.n - 1
        if qr < l or r < ql: return 0
        if ql <= l and r <= qr: return self.cnt[node]
        m = (l + r) // 2; self._push(node, l, m, r)
        return self.count(ql, qr, 2 * node, l, m) + self.count(ql, qr, 2 * node + 1, m + 1, r)

rnd_v = random.Random(15)
for _ in range(200):
    n = rnd_v.randint(1, 20); st = SegFlip(n); arr = [0] * n
    for _ in range(30):
        l = rnd_v.randrange(n); r = rnd_v.randrange(l, n)
        if rnd_v.random() < 0.5:
            st.flip(l, r)
            for i in range(l, r + 1): arr[i] ^= 1
        else:
            assert st.count(l, r) == sum(arr[l:r + 1])
print("Biến thể đảo đoạn (segment tree lazy): khớp mảng ngây thơ trên 200 chuỗi")

Biến thể đảo đoạn (segment tree lazy): khớp mảng ngây thơ trên 200 chuỗi


## 16. So sánh các cách làm

| Cách | Ý tưởng | `flip` | `count` | Bộ nhớ | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Ngây thơ | làm đúng nghĩa | `O(n)` | `O(n)` | `O(n)` | chuẩn đối chiếu |
| 2. Cờ lười + đếm | `phys ^ flipped` | `O(1)` | `O(1)` | `n` byte | phổ biến nhất |
| 3. Hai mảng bù | đổi tham chiếu | `O(1)` | `O(1)` | `2n` byte | **nộp bài** (toString nhanh) |
| 4. Số nguyên lớn | `x ^= mask` | `O(n/30)` | `O(1)` | `n/8` byte | gọn, n vừa phải |
| 5. Khối 64 bit | words + cờ | `O(1)` | `O(1)` | `n/8` byte | giống thư viện thật |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `size = 3`, lệnh `fix(0), flip(), fix(0), count(), flip(), toString()`. Kết quả các lệnh trả giá trị?

**Bài 2 (tính tay):** Với cách 2, sau chuỗi `flip(), fix(1), flip(), unfix(1)` trên `size = 3`, `phys`, `flipped`, `cnt` là gì?

**Bài 3 (điền TODO):** Thêm phương thức `get(idx) -> int` (giá trị bit thật) cho `Bitset` (code nộp).

**Bài 4 (tự viết lại):** Viết lại `Bitset` dùng **một** `bytearray` + cờ lười, nhưng `toString` dùng `bytes.translate` để chạy trong C.

**Bài 5 (mini challenge):** Thêm `setAll(v)` (gán mọi bit bằng `v` trong `O(1)`) vào một bitset có `fix/unfix/count/toString`. Gợi ý: lưu "giá trị nền" `base` và thời điểm `stamp` của lần gán toàn cục; mỗi bit lưu `(giá trị, thời điểm ghi)`. Kiểm bằng bản ngây thơ.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def bitset_get(bs, idx):
    # TODO: 1 nếu bs.bits[idx] == ord('1') else 0
    pass

# Bài 4 — TODO
class BitsetTranslate:
    def __init__(self, size):
        # TODO: bytearray(size), flipped, cnt
        pass

# Bài 5 — TODO
class BitsetSetAll:
    def __init__(self, size):
        # TODO: base, stamp, val[], when[], cnt
        pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `fix(0)` ⇒ `100`; `flip` ⇒ `011`.
- Mức 2: `fix(0)` ⇒ `111`; `count` ⇒ 3.
- Mức 3: `flip` ⇒ `000`; `toString` ⇒ `"000"`. Giá trị trả: `3`, `"000"`.

**Bài 2**
- Mức 1: `flip`: `flipped = 1`, `cnt = 3`.
- Mức 2: `fix(1)`: bit thật đã là 1 ⇒ không đổi; `flip`: `flipped = 0`, `cnt = 0`.
- Mức 3: `unfix(1)`: bit thật 0 ⇒ không đổi. `phys = 000`, `flipped = 0`, `cnt = 0`.

**Bài 3**
- Mức 1: Code nộp lưu ký tự `'0'`/`'1'`.
- Mức 2: So với 49.
- Mức 3: Kiểm bằng `toString()[idx]`.

**Bài 4**
- Mức 1: `bytes.maketrans(b"\x00\x01", b"01")` và bảng ngược `b"10"` khi `flipped`.
- Mức 2: `self.phys.translate(table).decode()`.
- Mức 3: Đối chiếu runner.

**Bài 5**
- Mức 1: Bit `i` thật = `val[i]` nếu `when[i] > stamp`, ngược lại `base`.
- Mức 2: `setAll(v)`: `base = v`, `stamp = clock`, `cnt = v * n`.
- Mức 3: `fix/unfix` tính bit thật cũ để cập nhật `cnt`, rồi ghi `val[i]`, `when[i] = clock`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
ops1 = [("fix", [0]), ("flip", []), ("fix", [0]), ("count", []), ("flip", []), ("toString", [])]
r1 = run(Bitset, 3, ops1)
print("Bài 1:", r1); assert r1 == [None, None, None, 3, None, "000"] == run(BitsetNaive, 3, ops1)

# Bài 2
b2 = BitsetLazy(3)
for name, a in [("flip", []), ("fix", [1]), ("flip", []), ("unfix", [1])]:
    getattr(b2, name)(*a)
print("Bài 2: phys =", list(b2.phys), "flipped =", b2.flipped, "cnt =", b2.cnt)
assert (list(b2.phys), b2.flipped, b2.cnt) == ([0, 0, 0], 0, 0)

# Bài 3
def bitset_get_sol(bs, idx):
    return 1 if bs.bits[idx] == 49 else 0

rnd = random.Random(3)
for _ in range(300):
    size = rnd.randint(1, 10); bs = Bitset(size)
    for name, a in random_ops(size, 20, rnd, max_tostring=20):
        getattr(bs, name)(*a)
        s = bs.toString()
        assert all(bitset_get_sol(bs, i) == int(s[i]) for i in range(size))
print("Bài 3 OK")

Bài 1: [None, None, None, 3, None, '000']
Bài 2: phys = [0, 0, 0] flipped = 0 cnt = 0
Bài 3 OK


In [24]:
# Bài 4
class BitsetTranslateSol(BitsetLazy):
    T0 = bytes.maketrans(b"\x00\x01", b"01")
    T1 = bytes.maketrans(b"\x00\x01", b"10")
    def toString(self):
        return self.phys.translate(self.T1 if self.flipped else self.T0).decode()

# Bài 5
class BitsetSetAllSol:
    def __init__(self, size):
        self.n = size; self.base = 0; self.stamp = 0; self.clock = 0
        self.val = [0] * size; self.when = [0] * size; self.cnt = 0
    def _get(self, i):
        return self.val[i] if self.when[i] > self.stamp else self.base
    def _write(self, i, v):
        self.clock += 1
        self.cnt += v - self._get(i)
        self.val[i] = v; self.when[i] = self.clock
    def fix(self, i): self._write(i, 1)
    def unfix(self, i): self._write(i, 0)
    def setAll(self, v):
        self.clock += 1; self.base = v; self.stamp = self.clock; self.cnt = v * self.n
    def count(self): return self.cnt
    def toString(self): return "".join(str(self._get(i)) for i in range(self.n))

class NaiveSetAll(BitsetNaive):
    def setAll(self, v):
        self.bits = [v] * len(self.bits)

rnd = random.Random(5)
for _ in range(500):
    size = rnd.randint(1, 10)
    ops = random_ops(size, 30, rnd, max_tostring=30)
    assert run(BitsetTranslateSol, size, ops) == run(BitsetNaive, size, ops)
    ops2 = []
    for _ in range(30):
        r = rnd.random()
        if r < 0.35: ops2.append(("fix", [rnd.randrange(size)]))
        elif r < 0.7: ops2.append(("unfix", [rnd.randrange(size)]))
        elif r < 0.8: ops2.append(("setAll", [rnd.randint(0, 1)]))
        elif r < 0.9: ops2.append(("count", []))
        else: ops2.append(("toString", []))
    assert run(BitsetSetAllSol, size, ops2) == run(NaiveSetAll, size, ops2)
print("Bài 4, Bài 5 OK")

Bài 4, Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "thao tác toàn cục lười + bộ đếm":**

```python
self.phys = bytearray(n); self.flipped = 0; self.cnt = 0
def real(i): return self.phys[i] ^ self.flipped
def fix(i):
    if not real(i): self.phys[i] ^= 1; self.cnt += 1
def flip(): self.flipped ^= 1; self.cnt = n - self.cnt
```

**Template hai mảng bù:** `bits, inv = inv, bits` khi `flip`; `fix/unfix` sửa **cả hai**.

**Checklist:**
- [ ] `fix` trên bit đã là 1 (và `unfix` trên bit 0) **không** đổi `cnt`?
- [ ] `flip` cập nhật `cnt = n − cnt`?
- [ ] Cờ lười được dùng ở **mọi** chỗ đọc/ghi (kể cả `toString`)?
- [ ] Hai mảng: `inv` luôn là phần bù?
- [ ] `all` ⇔ `cnt == n`, `one` ⇔ `cnt > 0`?
- [ ] Đã chạy runner đối chiếu bản ngây thơ + `check_invariant`?
- [ ] Đúng tên lớp/phương thức: `Bitset`, `fix`, `unfix`, `flip`, `all`, `one`, `count`, `toString`?